In [2]:
# %% [markdown]
# # Step 5: Verify Scalers for Web Application
# ## Check that all scalers are properly saved for the Flask app

# %%
import os
import pickle
import numpy as np

# %% [markdown]
# ### Verify scalers directory exists

# %%
# Create utils/scalers directory if it doesn't exist
os.makedirs('utils/scalers', exist_ok=True)
print("✅ Created/Verified utils/scalers directory")

# %% [markdown]
# ### Check if scalers are already saved

# %%
print("\n" + "="*50)
print("🔍 CHECKING FOR EXISTING SCALERS")
print("="*50)

# Define scaler files to check
scaler_files = {
    'continuous_scaler.pkl': 'Temperature scaler (StandardScaler for Avg_Temp and Soil)',
    'days_scaler.pkl': 'Days scaler (MinMaxScaler for Number_of_Days)',
    'nitrogen_map.pkl': 'Nitrogen mapping (fertilizer to categories)',
    'scaler_y.pkl': 'Target scaler (StandardScaler for Nitrogen_Cont)'
}

all_scalers_exist = True
existing_scalers = []
missing_scalers = []

for filename, description in scaler_files.items():
    filepath = os.path.join('utils/scalers', filename)
    if os.path.exists(filepath):
        file_size = os.path.getsize(filepath) / 1024  # KB
        existing_scalers.append(filename)
        print(f"✅ {filename} exists ({file_size:.2f} KB) - {description}")
    else:
        missing_scalers.append(filename)
        print(f"❌ {filename} missing - {description}")
        all_scalers_exist = False

# Check temperature means
temp_means_path = os.path.join('utils', 'temperature_means.pkl')
if os.path.exists(temp_means_path):
    file_size = os.path.getsize(temp_means_path) / 1024
    print(f"✅ temperature_means.pkl exists ({file_size:.2f} KB)")
else:
    print(f"❌ temperature_means.pkl missing")
    all_scalers_exist = False

# %% [markdown]
# ### If scalers are missing, provide instructions to create them

# %%
if not all_scalers_exist:
    print("\n" + "="*50)
    print("⚠️ SCALERS MISSING - INSTRUCTIONS")
    print("="*50)
    print("\nTo create the missing scalers, you need to:")
    print("1. Place your training Excel file in the 'data' folder")
    print("2. Update the path below to your training data")
    print("3. Run the code in the next cell")
    print("\nMissing scalers: " + ", ".join(missing_scalers))
    
    # Optional: Create a data folder if it doesn't exist
    os.makedirs('data', exist_ok=True)
    print("\n✅ Created 'data' folder for training data")
    
else:
    print("\n" + "="*50)
    print("✅ ALL SCALERS ARE PRESENT!")
    print("="*50)
    print("\nThe web application will use these scalers for predictions.")
    print("No further action needed.")

# %% [markdown]
# ### If scalers are missing, uncomment and run this cell to create them
# ### Place your Train.xlsx file in the 'data' folder first

# %%
# UNCOMMENT THE FOLLOWING CODE IF SCALERS ARE MISSING
"""
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Path to your training data (place Excel file in 'data' folder)
TRAIN_EXCEL_PATH = os.path.join('data', 'Train.xlsx')

if os.path.exists(TRAIN_EXCEL_PATH):
    try:
        # Load training data
        df = pd.read_excel(TRAIN_EXCEL_PATH, header=0)
        print(f"✅ Loaded {len(df)} samples from training data")
        
        # Extract features
        temperature_features = df[['Avg_Temp', 'Soil']].values.astype('float32')
        days = df['Number_of_Days'].values.reshape(-1, 1).astype('float32')
        nitrogen_content = df['Nitrogen_Cont'].values.reshape(-1, 1).astype('float32')
        
        # Create and fit temperature scaler (StandardScaler)
        continuous_scaler = StandardScaler()
        continuous_scaler.fit(temperature_features)
        print(f"✅ Fitted continuous_scaler on {temperature_features.shape[0]} samples")
        print(f"   Temperature means: {continuous_scaler.mean_}")
        
        # Create and fit days scaler (MinMaxScaler)
        days_scaler = MinMaxScaler()
        days_scaler.fit(days)
        print(f"✅ Fitted days_scaler on {days.shape[0]} samples")
        print(f"   Days range: [{days.min()}, {days.max()}] -> [{days_scaler.data_min_[0]:.2f}, {days_scaler.data_max_[0]:.2f}]")
        
        # Create and fit target scaler (StandardScaler for nitrogen)
        scaler_y = StandardScaler()
        scaler_y.fit(nitrogen_content)
        print(f"✅ Fitted scaler_y on {nitrogen_content.shape[0]} samples")
        print(f"   Nitrogen mean: {scaler_y.mean_[0]:.4f}, std: {scaler_y.scale_[0]:.4f}")
        
        # Save all scalers
        with open('utils/scalers/continuous_scaler.pkl', 'wb') as f:
            pickle.dump(continuous_scaler, f)
        print("✅ Saved continuous_scaler.pkl")
        
        with open('utils/scalers/days_scaler.pkl', 'wb') as f:
            pickle.dump(days_scaler, f)
        print("✅ Saved days_scaler.pkl")
        
        with open('utils/scalers/scaler_y.pkl', 'wb') as f:
            pickle.dump(scaler_y, f)
        print("✅ Saved scaler_y.pkl")
        
        # Save nitrogen mapping
        nitrogen_map = {0:0, 30:1, 60:2, 90:3, 120:4, 150:5, 180:6, 210:7}
        with open('utils/scalers/nitrogen_map.pkl', 'wb') as f:
            pickle.dump(nitrogen_map, f)
        print("✅ Saved nitrogen_map.pkl")
        
        # Save temperature means
        temperature_values = {'soil_temp_mean': 29.8}
        with open('utils/temperature_means.pkl', 'wb') as f:
            pickle.dump(temperature_values, f)
        print("✅ Saved temperature_means.pkl")
        
        print("\n✅ All scalers created successfully!")
        
    except Exception as e:
        print(f"❌ Error creating scalers: {e}")
        print("   Please check your Excel file format and columns")
else:
    print(f"❌ Training data not found at: {TRAIN_EXCEL_PATH}")
    print("   Please place your Train.xlsx file in the 'data' folder")
"""

# %% [markdown]
# ### Verify all scalers are valid

# %%
print("\n" + "="*50)
print("🔍 VERIFYING SCALER INTEGRITY")
print("="*50)

try:
    # Load and test scalers if they exist
    if all_scalers_exist:
        # Load scalers
        with open('utils/scalers/continuous_scaler.pkl', 'rb') as f:
            continuous_scaler = pickle.load(f)
        
        with open('utils/scalers/days_scaler.pkl', 'rb') as f:
            days_scaler = pickle.load(f)
        
        with open('utils/scalers/nitrogen_map.pkl', 'rb') as f:
            nitrogen_map = pickle.load(f)
        
        with open('utils/scalers/scaler_y.pkl', 'rb') as f:
            scaler_y = pickle.load(f)
        
        with open('utils/temperature_means.pkl', 'rb') as f:
            temp_means = pickle.load(f)
        
        print("✅ All scalers loaded successfully")
        
        # Test continuous scaler
        print("\n📊 Testing continuous_scaler (temperature):")
        test_temps = np.array([[20, 25], [25, 30], [30, 35]], dtype=np.float32)
        scaled_temps = continuous_scaler.transform(test_temps)
        print(f"   Input temps: {test_temps[0]} → Scaled: {scaled_temps[0]}")
        print(f"   Input temps: {test_temps[1]} → Scaled: {scaled_temps[1]}")
        print(f"   Input temps: {test_temps[2]} → Scaled: {scaled_temps[2]}")
        
        # Test days scaler
        print("\n📊 Testing days_scaler:")
        test_days = np.array([[60], [90], [120]], dtype=np.float32)
        scaled_days = days_scaler.transform(test_days)
        print(f"   Input days: {test_days[0][0]} → Scaled: {scaled_days[0][0]:.3f}")
        print(f"   Input days: {test_days[1][0]} → Scaled: {scaled_days[1][0]:.3f}")
        print(f"   Input days: {test_days[2][0]} → Scaled: {scaled_days[2][0]:.3f}")
        
        # Test nitrogen mapping
        print("\n📊 Testing nitrogen_map:")
        test_fertilizers = [0, 30, 60, 90, 120, 150, 180, 210]
        for fert in test_fertilizers:
            category = nitrogen_map.get(fert, 0)
            print(f"   Fertilizer {fert} kg/ha → Category {category}")
        
        # Test inverse transform
        print("\n📊 Testing scaler_y inverse transform:")
        test_predictions = np.array([[0.5], [1.0], [1.5]], dtype=np.float32)
        original_values = scaler_y.inverse_transform(test_predictions)
        for i, (pred, orig) in enumerate(zip(test_predictions, original_values)):
            print(f"   Normalized {pred[0]:.2f} → Original {orig[0]:.2f}%")
        
        # Test temperature means
        print("\n📊 Testing temperature_means:")
        print(f"   Soil Temperature Mean: {temp_means.get('soil_temp_mean', 'N/A')}°C")
        
        print("\n" + "="*50)
        print("✅ ALL SCALERS ARE VALID AND READY FOR USE!")
        print("="*50)
        
    else:
        print("\n⚠️ Cannot verify scalers because some are missing.")
        print("   Please create the missing scalers first.")
        
except Exception as e:
    print(f"❌ Verification failed: {e}")

# %% [markdown]
# ### Summary

# %%
print("\n" + "="*50)
print("📁 SCALER FILES SUMMARY")
print("="*50)

if all_scalers_exist:
    print("\n✅ All scaler files are present in utils/scalers/:")
    for filename in scaler_files.keys():
        filepath = os.path.join('utils/scalers', filename)
        if os.path.exists(filepath):
            file_size = os.path.getsize(filepath) / 1024
            print(f"   • {filename} ({file_size:.2f} KB)")
    
    if os.path.exists(temp_means_path):
        file_size = os.path.getsize(temp_means_path) / 1024
        print(f"   • temperature_means.pkl ({file_size:.2f} KB)")
    
    print("\n✅ The web application will use these scalers for:")
    print("   • Scaling temperature features (air and soil temperature)")
    print("   • Scaling days after sowing (60-120 days range)")
    print("   • Mapping fertilizer amounts to categories (0-7)")
    print("   • Denormalizing model predictions to actual nitrogen content")
    print("   • Providing soil temperature mean for missing values")
    
else:
    print("\n⚠️ Some scaler files are missing. Please create them using the instructions above.")
    print("\nRequired files for web application:")
    for filename in scaler_files.keys():
        print(f"   • utils/scalers/{filename}")
    print(f"   • utils/temperature_means.pkl")

print("\n" + "="*50)
print("🎉 SCALER SETUP COMPLETE!")
print("="*50)

✅ Created/Verified utils/scalers directory

🔍 CHECKING FOR EXISTING SCALERS
✅ continuous_scaler.pkl exists (0.49 KB) - Temperature scaler (StandardScaler for Avg_Temp and Soil)
✅ days_scaler.pkl exists (0.49 KB) - Days scaler (MinMaxScaler for Number_of_Days)
✅ nitrogen_map.pkl exists (0.05 KB) - Nitrogen mapping (fertilizer to categories)
✅ scaler_y.pkl exists (0.46 KB) - Target scaler (StandardScaler for Nitrogen_Cont)
✅ temperature_means.pkl exists (0.04 KB)

✅ ALL SCALERS ARE PRESENT!

The web application will use these scalers for predictions.
No further action needed.

🔍 VERIFYING SCALER INTEGRITY
✅ All scalers loaded successfully

📊 Testing continuous_scaler (temperature):
   Input temps: [20. 25.] → Scaled: [ 1.1058507  -0.14331882]
   Input temps: [25. 30.] → Scaled: [2.334751   0.69844556]
   Input temps: [30. 35.] → Scaled: [3.5636516 1.5402099]

📊 Testing days_scaler:
   Input days: 60.0 → Scaled: -0.070
   Input days: 90.0 → Scaled: 0.628
   Input days: 120.0 → Scaled: 1.3